# Stage 2: Instruction Fine-Tuning (SFT)
**Goal:** Teach the model to answer finance questions using 100 Q&A pairs.

## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

## Step 2 — Imports

In [ ]:
import torch
import json
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 3 — Configuration

In [ ]:
MODEL_NAME   = "unsloth/Llama-3.2-1B"   # Or path to Stage 1 adapter
MAX_SEQ_LEN  = 512
LOAD_IN_4BIT = True
LORA_R       = 16
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05
TARGET_MODS  = ["q_proj", "k_proj", "v_proj", "o_proj",
                "up_proj", "down_proj", "gate_proj"]
OUTPUT_DIR   = "./outputs/stage2_sft"
EPOCHS       = 3
BATCH_SIZE   = 2
GRAD_ACCUM   = 4
LR           = 2e-4

# Alpaca prompt template
ALPACA_TEMPLATE = """Below is a finance question. Write a clear, accurate, and helpful response.

### Question:
{}

### Answer:
{}"""

ALPACA_TEMPLATE_INFERENCE = """Below is a finance question. Write a clear, accurate, and helpful response.

### Question:
{}

### Answer:
"""

## Step 4 — Load model
> Option A: Load from base model
> Option B: Load from Stage 1 adapter (recommended)

In [ ]:
# OPTION A — Fresh base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)

# OPTION B — Continue from Stage 1 adapter (uncomment if using)
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name     = "./outputs/stage1_adapter",
#     max_seq_length = MAX_SEQ_LEN,
#     dtype          = None,
#     load_in_4bit   = LOAD_IN_4BIT,
# )

print("Model loaded")

## Step 5 — Apply LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = TARGET_MODS,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)
model.print_trainable_parameters()

## Step 6 — Load and format instruction dataset

In [ ]:
EOS = tokenizer.eos_token

# Load JSONL
records = []
with open("data/instruction_dataset.jsonl", "r") as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#"):
            records.append(json.loads(line))

print(f"Total examples loaded: {len(records)}")
print("Sample:", records[0])

# Format into Alpaca template
def format_example(rec):
    return ALPACA_TEMPLATE.format(rec["instruction"], rec["response"]) + EOS

formatted = [format_example(r) for r in records]
dataset = Dataset.from_dict({"text": formatted})
print(f"\nDataset size: {len(dataset)}")

## Step 7 — Train

In [ ]:
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    args = TrainingArguments(
        output_dir                  = OUTPUT_DIR,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate               = LR,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        save_strategy               = "epoch",
        warmup_ratio                = 0.1,
        lr_scheduler_type           = "cosine",
        report_to                   = "none",
    ),
)

print("Starting instruction fine-tuning (SFT)...")
trainer_stats = trainer.train()
print(f"Training complete. Final loss: {trainer_stats.training_loss:.4f}")

## Step 8 — Save SFT adapter

In [ ]:
SFT_ADAPTER_PATH = "./outputs/stage2_sft_adapter"
model.save_pretrained(SFT_ADAPTER_PATH)
tokenizer.save_pretrained(SFT_ADAPTER_PATH)
print(f"SFT adapter saved to: {SFT_ADAPTER_PATH}")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    model.save_pretrained("/content/drive/MyDrive/finance_ft/stage2_sft_adapter")
    print("SFT adapter also saved to Google Drive")
except:
    print("Not on Colab — skipping Drive save")

## Step 9 — Run inference on 10 test questions
> Save these outputs for sft_model_comparison.md

In [ ]:
FastLanguageModel.for_inference(model)

test_questions = [
    "What is a mutual fund SIP?",
    "What is the difference between a savings account and a fixed deposit?",
    "How does compound interest work?",
    "What is a credit score and why does it matter?",
    "How can I save tax under Section 80C?",
    "What is the repo rate and how does it affect loans?",
    "What is the difference between term insurance and whole life insurance?",
    "How does the stock market work?",
    "What is a demat account and how do I open one?",
    "What is the difference between a mutual fund and an ETF?",
]

sft_outputs = {}
for q in test_questions:
    prompt = ALPACA_TEMPLATE_INFERENCE.format(q)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = 150,
        temperature    = 0.3,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the answer part
    answer = answer.split("### Answer:")[-1].strip()
    sft_outputs[q] = answer
    print(f"Q: {q}")
    print(f"A: {answer[:200]}")
    print("-" * 60)